In [19]:
from dotenv import load_dotenv
import logging
import os
from pathlib import Path

import pandas as pd
import requests
from google import genai
import PIL.Image
from IPython.display import Image, display
from PIL import Image as PILImage


logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
log = logging.getLogger(__name__)


def load_dataset(csv_path: str = "observations.csv") -> pd.DataFrame:
    """Load the dataset CSV into a pandas DataFrame."""
    # tab seperated csv
    return pd.read_csv(csv_path)


def download_images(
    df: pd.DataFrame,
    output_dir: str | os.PathLike = "images",
    obs_id_col: str = "observation_id",
    photo_id_col: str = "photo_id",
) -> None:
    """Download images from URLs in the dataframe and save them to disk.

    Filenames are of the form: obs_<observation_id>_photo_<photo_id>.jpg

    For each row, the function looks for valid URLs in this order:
    1. ``original_url``
    2. ``large_url``
    3. ``medium_url``
    4. ``square_url``
    """
    output_path = Path(output_dir)

    for idx, row in df.iterrows():
        obs_id = row.get(obs_id_col)
        photo_id = row.get(photo_id_col)

        # using Image_name instead of obs_id and photo_id
        filename = f"{row['Image_name']}.jpg"
        img_path = output_path / filename

        # Skip if already downloaded
        if img_path.exists():
            log.info("Skipping %s because it already exists", filename)
            continue

        # Build candidate URLs in the required priority order
        candidate_urls = []
        for col in ("original_url", "large_url", "medium_url", "square_url"):
            candidate = row.get(col)
            if isinstance(candidate, str) and not pd.isna(candidate):
                candidate_urls.append(candidate)

        if not candidate_urls:
            log.warning("No valid URLs found for %s %s", obs_id, photo_id)
            continue

        img_bytes = None
        for candidate in candidate_urls:
            try:
                resp = requests.get(candidate, timeout=15)
                resp.raise_for_status()
                img_bytes = resp.content
                break
            except requests.RequestException:
                continue

        if img_bytes is None:
            # All attempts failed
            continue

        img_path.write_bytes(img_bytes)
        log.info("Wrote %s to directory %s",
                 filename, img_path.parent.resolve())


load_dotenv()
client = genai.Client()

PROMPT = "Generate a one sentence alt text for this image."

image_dir = "images"
results_path = "results/gemini_results.csv"


def generate_alt(df):
    done = set()
    if Path(results_path).exists():
        done = set(pd.read_csv(results_path)["Image_name"])
    else:
        pd.DataFrame(columns=["Image_name", "scientific_name", "gemini_alt_text"]).to_csv(
            results_path, index=False)

    for idx, row in df.iterrows():
        # Skip if already generated
        if row["Image_name"] in done:
            log.info("Skipping %s because alt text already exists",
                     row["Image_name"])
            continue

        img_path = Path(image_dir) / f"{row['Image_name']}.jpg"
        image = PIL.Image.open(img_path)
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=[PROMPT, image],
        )
        result = {"Image_name": row["Image_name"], "scientific_name": row["scientific_name"], "uri": row["uri"],
                  "Image_url": row["medium_url"], "gemini_alt_text": response.text.strip(), "model": "gemini-3.5-flash"}
        pd.DataFrame([result]).to_csv(
            results_path, mode='a', header=False, index=False)
        log.info("Done: %s", row['Image_name'])


if __name__ == "__main__":
    df = load_dataset()
    log.info("Loaded dataframe with shape: %s", df.shape)
    download_images(df)
    log.info("Image download complete.")
    generate_alt(df)
    log.info("Alt text generated!")


2026-06-28 14:07:47,776 - INFO - Loaded dataframe with shape: (63, 31)
2026-06-28 14:07:47,778 - INFO - Skipping obs_102691691_photo_171703626.jpg because it already exists
2026-06-28 14:07:47,778 - INFO - Skipping obs_10317797_photo_14298032.jpg because it already exists
2026-06-28 14:07:47,779 - INFO - Skipping obs_130480768_photo_70327211.jpg because it already exists
2026-06-28 14:07:47,779 - INFO - Skipping obs_188731534_photo_330410184.jpg because it already exists
2026-06-28 14:07:47,780 - INFO - Skipping obs_194599339_photo_342107100.jpg because it already exists
2026-06-28 14:07:47,780 - INFO - Skipping obs_194795499_photo_341224314.jpg because it already exists
2026-06-28 14:07:47,780 - INFO - Skipping obs_195713651_photo_344383957.jpg because it already exists
2026-06-28 14:07:47,781 - INFO - Skipping obs_220102252_photo_389649357.jpg because it already exists
2026-06-28 14:07:47,781 - INFO - Skipping obs_220822650_photo_391000325.jpg because it already exists
2026-06-28 14:

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 11.032131548s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'},

In [20]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv("results/gemini_results.csv")
for url in df["Image_url"].dropna().unique():
    display(Image(url=url, width=300))
display(df)


KeyError: 'Image_url'